In [1]:
!nvidia-smi

Wed Mar 11 15:54:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!ls '/content/drive/My Drive/Capstone/Cyclone-Data/yolo_custom_model_training'

custom_data.zip  Makefile  readme.txt  yolov3_custom.cfg


In [6]:
!unzip '/content/drive/My Drive/Capstone/Cyclone-Data/yolo_custom_model_training/custom_data.zip' -d '/content/drive/My Drive/Capstone/Cyclone-Data/yolo_custom_model_training'

Archive:  /content/drive/My Drive/Capstone/Cyclone-Data/yolo_custom_model_training/custom_data.zip
   creating: /content/drive/My Drive/Capstone/Cyclone-Data/yolo_custom_model_training/custom_data/
  inflating: /content/drive/My Drive/Capstone/Cyclone-Data/yolo_custom_model_training/custom_data/1.jpg  
  inflating: /content/drive/My Drive/Capstone/Cyclone-Data/yolo_custom_model_training/custom_data/1.txt  
  inflating: /content/drive/My Drive/Capstone/Cyclone-Data/yolo_custom_model_training/custom_data/10.jpg  
  inflating: /content/drive/My Drive/Capstone/Cyclone-Data/yolo_custom_model_training/custom_data/10.txt  
  inflating: /content/drive/My Drive/Capstone/Cyclone-Data/yolo_custom_model_training/custom_data/100.jpg  
  inflating: /content/drive/My Drive/Capstone/Cyclone-Data/yolo_custom_model_training/custom_data/100.txt  
  inflating: /content/drive/My Drive/Capstone/Cyclone-Data/yolo_custom_model_training/custom_data/101.jpg  
  inflating: /content/drive/My Drive/Capstone/Cyclon

In [7]:
!git clone 'https://github.com/AlexeyAB/darknet.git' '/content/drive/My Drive/Capstone/Cyclone-Data/yolo_custom_model_training/darknet'

Cloning into '/content/drive/My Drive/Capstone/Cyclone-Data/yolo_custom_model_training/darknet'...
remote: Enumerating objects: 15941, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 15941 (delta 3), reused 2 (delta 1), pack-reused 15927 (from 2)
Receiving objects: 100% (15941/15941), 14.50 MiB | 10.13 MiB/s, done.
Resolving deltas: 100% (10722/10722), done.
Updating files: 100% (2054/2054), done.


In [8]:
%cd /content/drive/My Drive/Capstone/Cyclone-Data/yolo_custom_model_training/darknet

/content/drive/My Drive/Capstone/Cyclone-Data/yolo_custom_model_training/darknet


In [9]:
!ls

3rdparty		data		       net_cam_v3.sh
build			docker-compose.yml     net_cam_v4.sh
CCM			Dockerfile.cpu	       package.xml
cfg			Dockerfile.gpu	       README.md
cmake			image_yolov3.sh        scripts
CMakeLists.txt		image_yolov4.sh        src
DarknetConfig.cmake.in	include		       vcpkg.json
darknet_images.py	json_mjpeg_streams.sh  video_yolov3.sh
darknet.py		LICENSE		       video_yolov4.sh
darknet_video.py	Makefile


In [10]:
!make

mkdir -p ./obj/
mkdir -p backup
mkdir -p results
chmod +x *.sh
g++ -std=c++11 -std=c++11 -Iinclude/ -I3rdparty/stb/include -Wall -Wfatal-errors -Wno-unused-result -Wno-unknown-pragmas -fPIC -rdynamic -Ofast -c ./src/image_opencv.cpp -o obj/image_opencv.o
g++ -std=c++11 -std=c++11 -Iinclude/ -I3rdparty/stb/include -Wall -Wfatal-errors -Wno-unused-result -Wno-unknown-pragmas -fPIC -rdynamic -Ofast -c ./src/http_stream.cpp -o obj/http_stream.o
./src/http_stream.cpp: In member function ‘bool JSON_sender::write(const char*)’:
./src/http_stream.cpp:253:21: warning: unused variable ‘n’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wunused-variable-Wunused-variable]8;;]
  253 |                 int n = _write(client, outputbuf, outlen);
      |                     ^
./src/http_stream.cpp: In function ‘void set_track_id(detection*, int, float, float, float, int, int, int)’:
./src/http_stream.cpp:866:27: warning: comparison of integer expressions of different signedness: 

In [11]:
%cd /content/drive/My Drive/Capstone/Cyclone-Data/yolo_custom_model_training/darknet
!sed -i 's/OPENCV=0/OPENCV=1/' Makefile
!sed -i 's/GPU=0/GPU=1/' Makefile
!sed -i 's/CUDNN=0/CUDNN=1/' Makefile

/content/drive/My Drive/Capstone/Cyclone-Data/yolo_custom_model_training/darknet


In [17]:
%cd '/content/drive/My Drive/Capstone/Cyclone-Data/yolo_custom_model_training'

/content/drive/My Drive/Capstone/Cyclone-Data/yolo_custom_model_training


In [30]:
# Copy data to writable directory (/tmp)
!cp -r '/content/drive/My Drive/Capstone/Cyclone-Data/yolo_custom_model_training/custom_data' /tmp/custom_data
%cd /tmp

# Run the setup scripts in /tmp
!python custom_data/creating-files-data-and-name.py
!python custom_data/creating-train-and-test-txt-files.py

# Copy darknet and weights to /tmp for training
!cp -r '/content/drive/My Drive/Capstone/Cyclone-Data/yolo_custom_model_training/darknet' /tmp/darknet
!mkdir -p /tmp/custom_weight
!cp '/content/drive/My Drive/Capstone/Cyclone-Data/yolo_custom_model_training/custom_weight/darknet53.conv.74' /tmp/custom_weight/

# Copy custom cfg file
!cp '/content/drive/My Drive/Capstone/Cyclone-Data/yolo_custom_model_training/yolov3_custom.cfg' /tmp/darknet/cfg/yolov3_custom.cfg

# Verify files were created
!ls -la /tmp/custom_data/classes.names
!ls -la /tmp/custom_data/labelled_data.data
!ls -la /tmp/darknet/cfg/yolov3_custom.cfg

/tmp
cp: cannot overwrite non-directory '/tmp/darknet/darknet' with directory '/content/drive/My Drive/Capstone/Cyclone-Data/yolo_custom_model_training/darknet'
cp: cannot stat '/content/drive/My Drive/Capstone/Cyclone-Data/yolo_custom_model_training/custom_weight/darknet53.conv.74': No such file or directory
-rw-r--r-- 1 root root 8 Mar 11 16:22 /tmp/custom_data/classes.names
-rw-r--r-- 1 root root 120 Mar 11 16:22 /tmp/custom_data/labelled_data.data
-rw------- 1 root root 9117 Mar 11 16:22 /tmp/darknet/cfg/yolov3_custom.cfg


In [24]:
!ls

custom_data
custom_weight
dap_multiplexer.83a817d02ebe.root.log.INFO.20260311-155204.115
dap_multiplexer.INFO
darknet
debugger_284vjh41ba
directoryprefetcher_binary.83a817d02ebe.root.log.INFO.20260311-155439.983
directoryprefetcher_binary.INFO
drive.83a817d02ebe.root.log.ERROR.20260311-155433.804
drive.83a817d02ebe.root.log.ERROR.20260311-155438.897
drive.83a817d02ebe.root.log.INFO.20260311-155432.793
drive.83a817d02ebe.root.log.INFO.20260311-155432.804
drive.83a817d02ebe.root.log.INFO.20260311-155437.793
drive.83a817d02ebe.root.log.INFO.20260311-155437.897
drive.83a817d02ebe.root.log.WARNING.20260311-155433.804
drive.83a817d02ebe.root.log.WARNING.20260311-155438.897
drive.ERROR
drivefs_ipc.0
drivefs_ipc.0_shell
drive.INFO
drive.WARNING
initgoogle_syslog_dir.0
language_service.83a817d02ebe.root.log.INFO.20260311-155416.690
language_service.INFO
pyright-697-EZomTsLR7Fmy
python-languageserver-cancellation


In [25]:
!chmod +x ./darknet/darknet

In [29]:
%cd /tmp
!chmod +x ./darknet/darknet
!/tmp/darknet/darknet detector train custom_data/labelled_data.data darknet/cfg/yolov3_custom.cfg custom_weight/darknet53.conv.74 -dont_show

/tmp
 GPU isn't used 
 OpenCV isn't used - data augmentation will be slow 
yolov3_custom
mini_batch = 1, batch = 4, time_steps = 1, train = 1 
   layer   filters  size/strd(dil)      input                output
   0 conv     32       3 x 3/ 1    416 x 416 x   3 ->  416 x 416 x  32 0.299 BF
   1 conv     64       3 x 3/ 2    416 x 416 x  32 ->  208 x 208 x  64 1.595 BF
   2 conv     32       1 x 1/ 1    208 x 208 x  64 ->  208 x 208 x  32 0.177 BF
   3 conv     64       3 x 3/ 1    208 x 208 x  32 ->  208 x 208 x  64 1.595 BF
   4 Shortcut Layer: 1,  wt = 0, wn = 0, outputs: 208 x 208 x  64 0.003 BF
   5 conv    128       3 x 3/ 2    208 x 208 x  64 ->  104 x 104 x 128 1.595 BF
   6 conv     64       1 x 1/ 1    104 x 104 x 128 ->  104 x 104 x  64 0.177 BF
   7 conv    128       3 x 3/ 1    104 x 104 x  64 ->  104 x 104 x 128 1.595 BF
   8 Shortcut Layer: 5,  wt = 0, wn = 0, outputs: 104 x 104 x 128 0.001 BF
   9 conv     64       1 x 1/ 1    104 x 104 x 128 ->  104 x 104 x  64 0.177 BF